In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [ ]:
!pip install transformers datasets -q

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6")
model = AutoModelForSequenceClassification.from_pretrained(
    "zhihan1996/DNA_bert_6", 
    num_labels=2
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/359M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/359M [00:00<?, ?B/s]

In [4]:
def sequence_to_kmer(sequence, k=6):
    kmer_list=[]
    for i in range(len(sequence)-k+1):
        kmer_list.append(sequence[i:i+k])

    return " ".join(kmer_list)

def clean_sequence(seq):
    seq = seq.upper()
    seq = seq.strip()

    return seq

In [5]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Load UCI promoter dataset directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"

df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])
df_cleaned = df.copy()
df_cleaned['label'] = df_cleaned['label'].map({'-': 0, '+': 1})
df_cleaned['sequence'] = df_cleaned['sequence'].apply(clean_sequence)
df_cleaned.head(3)

,label,id,sequence
0,1,S10,TACTAGCAATACGCTTGCGTTCGGTGGTTAAGTATGTATAATGCGC...
1,1,AMPC,TGCTATCCTGACAGTTGTCACGCTGATTGGTGTCGTTACAATCTAA...
2,1,AROH,GTACTAGAGAACTAGTGCATTAGCTTATTTTTTTGTTATCATGCTA...


In [6]:
tokenized_data = []
for sid in df_cleaned['id']:
    label = df_cleaned.label[df['id']==sid].item()
    seq_tokens = sequence_to_kmer(df_cleaned.sequence[df['id']==sid].item())
    tokenized = tokenizer(seq_tokens, return_tensors='pt', padding='max_length', max_length=64)
    tokenized_data.append({"input_ids": tokenized["input_ids"],
                          "attention_mask": tokenized["attention_mask"],
                          "label": label})

In [8]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
sequences = df_cleaned['sequence'].values
labels = df_cleaned['label'].values

fold_accuracies = []
fold_losses = []

for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels)):
    
    print(f"{'*'*15}Fold {fold + 1} results {'*'*15}")
    # 1. Split data using train_idx, test_idx
    train_sequences, train_labels = sequences[train_idx], labels[train_idx]
    test_sequences, test_labels = sequences[test_idx], labels[test_idx]

    
    # 2. Create FRESH model + optimizer + criterion
    model = AutoModelForSequenceClassification.from_pretrained(
        "zhihan1996/DNA_bert_6", 
        num_labels=2,
        ignore_mismatched_sizes=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    # 3. Train for 3 epochs
    
    for epoch in range(10): 
        epoch_loss = 0       
        model.train()
        for i, X_train in enumerate(train_sequences):
           tokenized = tokenizer(sequence_to_kmer(X_train), return_tensors='pt', padding='max_length', max_length=64)
           label = torch.tensor(train_labels[i]).unsqueeze(-1)
           outputs = model(input_ids=tokenized["input_ids"],
                        attention_mask=tokenized["attention_mask"],
                        labels=label)
           loss = outputs.loss
           epoch_loss += loss.item()
           loss.backward()
           optimizer.step()
           optimizer.zero_grad()

    print(f"Training loss (averaged across epochs): {epoch_loss/20}")

    # 6. Evaluate on test fold
    model.eval()
    correct = 0
    total = len(test_sequences)
    with torch.no_grad():
        final_loss = 0
        for i, X_test in enumerate(test_sequences):
            tokenized = tokenizer(sequence_to_kmer(X_test), return_tensors='pt', padding='max_length', max_length=64)
            label = torch.tensor(test_labels[i]).unsqueeze(-1)           
            outputs = model(input_ids=tokenized["input_ids"],
                        attention_mask=tokenized["attention_mask"],
                        labels=label)
            logits = outputs.logits
            loss = outputs.loss
            final_loss += loss.item()
            prediction = logits.argmax().item()
            if prediction == label:
                correct += 1
            
    print(f"  Final test fold loss:  {final_loss/len(test_sequences):.4f}")
    # 7. Append accuracy to fold_accuracies        
    accuracy = correct / total if total > 0 else 0
    fold_accuracies.append(accuracy)

print(f"\n{'='*40}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*40}")
print(f"Fold accuracies: {[f'{a:.2%}' for a in fold_accuracies]}")
print(f"Mean accuracy:   {np.mean(fold_accuracies):.2%}")
print(f"Std deviation:   {np.std(fold_accuracies):.2%}")

***************Fold 1 results ***************


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training loss (averaged across epochs): 0.29879334768629634
  Final test fold loss:  0.5511
***************Fold 2 results ***************


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training loss (averaged across epochs): 0.215318580949679
  Final test fold loss:  0.4384
***************Fold 3 results ***************


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training loss (averaged across epochs): 0.057587204361334444
  Final test fold loss:  0.4287

CROSS-VALIDATION RESULTS
Fold accuracies: ['66.67%', '82.86%', '85.71%']
Mean accuracy:   78.41%
Std deviation:   8.39%


In [23]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
model.train()

for epoch in range(10):
    epoch_loss = 0
    for sample in tokenized_data:
        label = torch.tensor([sample["label"]])
        outputs = model(input_ids=sample["input_ids"],
                        attention_mask=sample["attention_mask"],
                        labels=label)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(tokenized_data):.4f}")

Epoch 1, Loss: 0.4426
Epoch 2, Loss: 0.4491
Epoch 3, Loss: 0.3091
Epoch 4, Loss: 0.1845
Epoch 5, Loss: 0.1060
Epoch 6, Loss: 0.0909
Epoch 7, Loss: 0.0608
Epoch 8, Loss: 0.1051
Epoch 9, Loss: 0.0763
Epoch 10, Loss: 0.0230
